In [ ]:
import torch
from huggingface_hub import hf_hub_download

hf_token = ''

repo_id = "google/gemma-4-26B-A4B-it-qat-q4_0-gguf" # 14.4GB
filename = "gemma-4-26B_q4_0-it.gguf"

# repo_id = "google/gemma-4-12B-it-qat-q4_0-gguf"
# filename = "gemma-4-12b-it-qat-q4_0.gguf"

# MODEL_ID = "google/gemma-4-12B-it-qat-w4a16-ct"

print("Downloading model...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename, token=hf_token)
print(f"Model downloaded and saved to: {model_path}")

print(f"Current GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA Available?: {torch.cuda.is_available()}")

In [ ]:
from llama_cpp import Llama

# n_ctx should handle your prompt length; Gemma 4 supports up to 256k, but 2048/4096 is standard for local classification

token_max = 8192

llm = Llama(
    model_path=model_path,
    n_ctx=token_max,  
    n_gpu_layers=-1, # Passes all layers to GPU/Metal (-1 means max offload)
    verbose=False
)

In [ ]:
import re
import json
from llama_cpp import LlamaGrammar

categories = []
json_file_path = 'categories.json'
with open(json_file_path, 'r') as f:
    categories_data = json.load(f)
    
for mapping in categories_data:
    for category in mapping.get('categories'):
        categories.append(category)

print(categories)

cat_rules = []
cat_choices = []

for c in categories:
    # Rule names must only contain basic alpha characters or hyphens
    # "Food & Dining" becomes "cat-Food-Dining"
    rule_name = "cat-" + re.sub(r'[^a-zA-Z0-9-]', '', c.replace(" ", "-"))
    
    # Define the rule to match the literal category string with quotes
    cat_rules.append(f'{rule_name} ::= "\\"{c}\\""')
    cat_choices.append(rule_name)

# Join the options with the pipe (|) separator
category_choices_str = " | ".join(cat_choices)
rules_definition_str = "\n".join(cat_rules)

gbnf_grammar = f"""
root      ::= "{{" ws pair (ws "," ws pair)? (ws "," ws pair)? "}}"
pair      ::= category ws ":" ws score
category  ::= {category_choices_str}
score     ::= "0." [0-9] [0-9]? | "1.0"
ws        ::= [ \\t\\n\\r]*

{rules_definition_str}
"""

compiled_grammar = LlamaGrammar.from_string(gbnf_grammar)

In [ ]:
def classify_text(text_to_classify, grammar_object):
    prompt = f"""<|im_start|>system
You are a text classification assistant. Analyze the text and identify up to 3 distinct categories that apply. 
Provide a confidence score (0.0 to 1.0) for each. 

CRITICAL: Do not repeat categories. Each category must be completely unique.
Return your response as a single JSON object where the keys are the categories and values are scores.
List the categories in order of confidence, highest score first.
Example: {{"Technology": 0.95, "Finance": 0.45}}<|im_end|>
<|im_start|>user
{text_to_classify[:(token_max-600)]}<|im_end|>
<|im_start|>assistant
"""

    response = llm(
        prompt,
        max_tokens=512,
        temperature=0.0,         # 0.0 ensures deterministic extraction
        grammar=grammar_object,
        repeat_penalty=1.3,
        stop=["<|im_end|>"]
    )
    
    # Extract text block
    output_text = response['choices'][0]['text'].strip()
    
    # Safeguard JSON Parsing
    try:
        result = []
        parsed_json = json.loads(output_text)
        
        for cat in parsed_json:
            if cat in categories:
                result.append((cat, parsed_json[cat]))
            # else:
                # return f"Error: Model generated an off-list category: {predicted_category}"
        return sorted(result, key=lambda x: x[1], reverse=True)
    except json.JSONDecodeError:
        return []

In [ ]:
def batch_classify_proposals(dataframe, text_column, grammar_obj):
    results = []
    
    for text in tqdm(dataframe[text_column], desc="Classifying Proposals"):
        try:
            classification_output = classify_text(text, grammar_obj)

            results.append(classification_output)
        except Exception as e:
            # Safeguard in case an individual row completely fails execution
            print(f"Error processing text: {e}")
            results.append([])
            
    return results

In [ ]:
from pyutil.db_entities import *
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

db_file = 'compiled.db'
eng = create_engine(f'sqlite:///{db_file}')
Base.metadata.create_all(eng)
session = Session(eng)

In [ ]:
import pandas as pd
from sqlalchemy import select, alias

projects = session.scalars(select(Project)).all()

project_revisions = []

for i, project in enumerate(projects):
    print(f"Processing Project: {project.project_name} (ID: {project.project_id})")
    
    pr, pr2 = alias(ProposalRevision, name="pr"), alias(ProposalRevision, name="pr2")
    stmt = (select(pr)
            .outerjoin(pr2, (pr.c.proposal_id == pr2.c.proposal_id) & (pr.c.revision_index < pr2.c.revision_index))
            .where(pr.c.project_id == project.project_id)
            .where(pr2.c.proposal_id == None)  # Only get the latest revision
        )

    revisions = session.execute(stmt).all()
    print(f"  Found {len(revisions)} unique proposal revisions for this project.")

    project_revisions.append(revisions)

#print revs excluding content for verification
print("Sample project revisions (excluding content):")
for revs in project_revisions:
    for rev in revs[:1]:
        print(f"  Project ID: {rev.project_id}, Proposal ID: {rev.proposal_id}, Revision Index: {rev.revision_index}, Created At: {rev.created_at}, Title: {rev.title}")

df = pd.DataFrame({
    'project_id': [rev.project_id for revs in project_revisions for rev in revs],
    'project_name': [projects[rev.project_id].project_name for revs in project_revisions for rev in revs],
    'proposal_id': [rev.proposal_id for revs in project_revisions for rev in revs],
    'revision_index': [rev.revision_index for revs in project_revisions for rev in revs],
    'created_at': [rev.created_at for revs in project_revisions for rev in revs],
    'title': [rev.title for revs in project_revisions for rev in revs],
    'content': [rev.content for revs in project_revisions for rev in revs]
})

print(f"Total projects processed: {len(project_revisions)}")
print(f"Total unique proposal revisions collected: {sum(len(revs) for revs in project_revisions)}")

In [ ]:
from tqdm import tqdm

raw_results = batch_classify_proposals(df, 'content', compiled_grammar)

In [ ]:
import numpy as np

cooked_results = raw_results.copy()

for i, result in enumerate(cooked_results):
    if isinstance(result, str):
        cooked_results[i] = [("Documentation & Education", 0)]

assigned_categories = [[tup[0] for tup in sublist] for sublist in cooked_results]
assigned_scores = [[tup[1] for tup in sublist] for sublist in cooked_results]


In [ ]:
df_copy = df.copy()
df_copy['assigned_categories'] = assigned_categories
df_copy['assigned_scores'] = assigned_scores
df_copy.drop(columns=['content'], inplace=True)
df_copy.to_csv('classified_proposals_llm.csv', index=False)

print(df_copy.iloc[0])
df_copy.head()